# 🌿 Détection des Maladies des Plantes — Deep Learning
## ÉTAPE 3/9 : CNN From Scratch

**Dataset :** PlantVillage — 54 305 images RGB (224×224×3), 38 classes
**Objectif :** construire, entraîner et évaluer un **CNN développé entièrement à partir de zéro** (aucun poids pré-entraîné), qui servira de **modèle de référence (baseline)** pour le benchmarking de l'étape 4 face à MobileNetV2 et ResNet50.

---

### 📋 Plan du notebook

| Partie | Contenu |
|--------|---------|
| 0 | Reconstruction des générateurs (artefacts de l'étape 2) |
| 1 | Architecture du CNN (4 blocs convolutifs + classification) |
| 2 | Compilation (Adam, categorical_crossentropy, accuracy) |
| 3 | Callbacks (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint) |
| 4 | Entraînement (20 epochs, batch_size=32) |
| 5 | Évaluation (Accuracy, Loss, Precision, Recall, F1) |
| 6 | Visualisations (courbes, matrice de confusion, classification report) |
| 7 | Sauvegarde (`.keras` et `.h5`) |
| 8 | Analyse complète (forces / faiblesses / overfitting / recommandations) |

> 💡 Chaque partie est précédée d'une explication. Les lignes de code les plus importantes sont commentées directement dans les cellules.


## 0. ⚙️ Configuration et reconstruction des générateurs

**Pourquoi cette partie ?**
Tu as précisé que `train_generator`, `validation_generator` et `test_generator` existent déjà — c'est vrai **si tu exécutes ce notebook dans la même session** que celui de l'étape 2. Mais un notebook Colab/Kaggle peut être redémarré (perte de la mémoire RAM), donc, par sécurité et pour que **ce notebook fonctionne de façon autonome**, on **recharge les artefacts sauvegardés à l'étape 2** (`preprocessing_artifacts/`) et on **reconstruit les générateurs à l'identique** (mêmes splits, même normalisation `1/255`, même augmentation sur le train).

> 📂 Assure-toi que le dossier `preprocessing_artifacts/` (généré à l'étape 2) est présent dans le répertoire de travail. Sur Kaggle/Colab, si tu repars d'une session fraîche, ré-exécute d'abord le notebook de l'étape 2, ou uploade ce dossier.


In [ ]:
# Installation silencieuse
!pip install -q tensorflow scikit-learn seaborn pillow

import os
import json
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Vérification GPU
gpu_devices = tf.config.list_physical_devices("GPU")
print(f"TensorFlow version : {tf.__version__}")
print(f"GPU détecté(s) : {gpu_devices if gpu_devices else '⚠️ AUCUN — active le GPU (Runtime > Change runtime type)'}")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)


In [ ]:
SAVE_DIR = "preprocessing_artifacts"

# --- Chargement des artefacts sauvegardés à l'étape 2 ---
with open(os.path.join(SAVE_DIR, "class_names.json"), "r", encoding="utf-8") as f:
    class_info = json.load(f)

with open(os.path.join(SAVE_DIR, "preprocessing_config.json"), "r", encoding="utf-8") as f:
    preprocessing_config = json.load(f)

class_names = class_info["class_names"]
num_classes = class_info["num_classes"]

train_df = pd.read_csv(os.path.join(SAVE_DIR, "train_df.csv"))
val_df   = pd.read_csv(os.path.join(SAVE_DIR, "val_df.csv"))
test_df  = pd.read_csv(os.path.join(SAVE_DIR, "test_df.csv"))

IMG_SIZE = tuple(preprocessing_config["img_size"])
BATCH_SIZE = preprocessing_config["batch_size"]

print(f"✅ Artefacts chargés : {num_classes} classes")
print(f"   Train      : {len(train_df)} images")
print(f"   Validation : {len(val_df)} images")
print(f"   Test       : {len(test_df)} images")
print(f"   IMG_SIZE = {IMG_SIZE} | BATCH_SIZE = {BATCH_SIZE}")


In [ ]:
# --- Reconstruction des générateurs (configuration IDENTIQUE à l'étape 2) ---

# Train : normalisation [0,1] + augmentation (rotation, zoom, flip, shift, brightness)
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest",
)

# Validation / Test : normalisation SEULEMENT, aucune augmentation
eval_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

common_args = dict(
    x_col="filepath",
    y_col="label",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    seed=SEED,
)

train_generator = train_datagen.flow_from_dataframe(train_df, shuffle=True, **common_args)
validation_generator = eval_datagen.flow_from_dataframe(val_df, shuffle=False, **common_args)
test_generator = eval_datagen.flow_from_dataframe(test_df, shuffle=False, **common_args)

# Vérification de cohérence avec les class_names sauvegardés
assert train_generator.class_indices == class_info["class_indices"], "⚠️ Incohérence d'encodage des classes !"
print("✅ train_generator / validation_generator / test_generator reconstruits avec succès.")
print(f"✅ Encodage des {num_classes} classes vérifié et cohérent avec l'étape 2.")


## 🏗️ PARTIE 1 — Architecture du CNN From Scratch

### Logique de conception

| Élément | Rôle |
|---|---|
| **Conv2D** | Extrait des motifs visuels (bords, textures, taches de maladie...) via des filtres convolutifs |
| **BatchNormalization** | Stabilise et accélère l'entraînement en normalisant les activations à chaque couche |
| **MaxPooling2D** | Réduit la dimension spatiale (divise par 2), diminue le coût de calcul et rend le modèle plus robuste aux petites translations |
| **Nombre de filtres croissant (32→64→128→256)** | Les premières couches détectent des motifs simples (contours), les couches profondes détectent des motifs complexes (formes de lésions, textures de maladies) |
| **Flatten** | Transforme la carte de caractéristiques 2D en un vecteur 1D exploitable par les couches denses |
| **Dense(512)** | Couche entièrement connectée qui combine les caractéristiques extraites pour la décision finale |
| **Dropout(0.5)** | Désactive aléatoirement 50% des neurones à chaque itération d'entraînement → réduit le surapprentissage (overfitting) |
| **Dense(38, softmax)** | Couche de sortie : produit une probabilité pour chacune des 38 classes (somme = 1) |

> 📐 Avec une entrée de 224×224, après 4 MaxPooling (÷2 chacun), la carte de caractéristiques finale est de **14×14** — c'est pourquoi 4 blocs est un choix raisonnable (ni trop peu de réduction, ni une carte trop petite avant le Flatten).


In [ ]:
def build_cnn_from_scratch(input_shape=(224, 224, 3), num_classes=38):
    \"\"\"Construit le CNN From Scratch selon l'architecture spécifiée dans le cahier des charges.\"\"\"

    model = models.Sequential(name="CNN_From_Scratch_PlantDisease")

    # --- Couche d'entrée ---
    model.add(layers.Input(shape=input_shape))

    # --- BLOC 1 : Conv2D(32) -> BatchNorm -> MaxPooling ---
    # 32 filtres 3x3 : détecte des motifs simples (bords, contours, couleurs dominantes)
    model.add(layers.Conv2D(32, kernel_size=(3, 3), padding="same", activation="relu", name="block1_conv"))
    model.add(layers.BatchNormalization(name="block1_bn"))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name="block1_pool"))   # 224x224 -> 112x112

    # --- BLOC 2 : Conv2D(64) -> BatchNorm -> MaxPooling ---
    model.add(layers.Conv2D(64, kernel_size=(3, 3), padding="same", activation="relu", name="block2_conv"))
    model.add(layers.BatchNormalization(name="block2_bn"))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name="block2_pool"))   # 112x112 -> 56x56

    # --- BLOC 3 : Conv2D(128) -> BatchNorm -> MaxPooling ---
    model.add(layers.Conv2D(128, kernel_size=(3, 3), padding="same", activation="relu", name="block3_conv"))
    model.add(layers.BatchNormalization(name="block3_bn"))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name="block3_pool"))   # 56x56 -> 28x28

    # --- BLOC 4 : Conv2D(256) -> BatchNorm -> MaxPooling ---
    model.add(layers.Conv2D(256, kernel_size=(3, 3), padding="same", activation="relu", name="block4_conv"))
    model.add(layers.BatchNormalization(name="block4_bn"))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name="block4_pool"))   # 28x28 -> 14x14

    # --- CLASSIFICATION ---
    model.add(layers.Flatten(name="flatten"))                              # 14x14x256 -> vecteur de 50 176 valeurs
    model.add(layers.Dense(512, activation="relu", name="dense_512"))      # Couche dense de décision
    model.add(layers.Dropout(0.5, name="dropout_50"))                      # Régularisation anti-overfitting
    model.add(layers.Dense(num_classes, activation="softmax", name="output"))  # Sortie : probabilités sur 38 classes

    return model


cnn_model = build_cnn_from_scratch(input_shape=(*IMG_SIZE, 3), num_classes=num_classes)
cnn_model.summary()


In [ ]:
# Visualisation graphique de l'architecture (utile pour le rapport PDF / la présentation PowerPoint)
tf.keras.utils.plot_model(
    cnn_model,
    to_file="cnn_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    rankdir="TB",
    dpi=120,
)
print("💾 Schéma de l'architecture sauvegardé : cnn_architecture.png")

# Nombre total de paramètres entraînables (information clé pour le rapport)
total_params = cnn_model.count_params()
trainable_params = sum(np.prod(v.shape) for v in cnn_model.trainable_variables)
print(f"\n📊 Nombre total de paramètres     : {total_params:,}")
print(f"📊 Nombre de paramètres entraînables : {int(trainable_params):,}")


## ⚙️ PARTIE 2 — Compilation du modèle

**Pourquoi ces choix ?**
- **Optimizer `Adam`** : combine les avantages de Momentum et RMSProp, converge rapidement et nécessite peu de réglage manuel — c'est le choix standard pour démarrer un entraînement CNN.
- **Loss `categorical_crossentropy`** : adaptée à la classification **multi-classes avec labels one-hot** (38 classes, une seule classe correcte par image) — cohérent avec `class_mode="categorical"` des générateurs.
- **Metric `accuracy`** : métrique de suivi en temps réel pendant l'entraînement ; les métriques plus fines (Precision/Recall/F1) seront calculées en détail à la Partie 5.


In [ ]:
cnn_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),   # taux d'apprentissage initial standard
    loss="categorical_crossentropy",                   # adapté au one-hot encoding multi-classes
    metrics=["accuracy"],
)

print("✅ Modèle compilé : optimizer=Adam, loss=categorical_crossentropy, metrics=['accuracy']")


## 🛎️ PARTIE 3 — Callbacks

**Pourquoi ces 3 callbacks ?**

| Callback | Rôle | Paramètres choisis |
|---|---|---|
| **EarlyStopping** | Arrête l'entraînement si `val_loss` ne s'améliore plus, pour éviter de continuer à overfitter inutilement | `patience=5`, `restore_best_weights=True` → on récupère automatiquement les meilleurs poids, pas ceux du dernier epoch |
| **ReduceLROnPlateau** | Réduit le taux d'apprentissage quand l'entraînement stagne, pour permettre un affinage plus fin | `factor=0.5`, `patience=3` → divise le LR par 2 si pas d'amélioration pendant 3 epochs |
| **ModelCheckpoint** | Sauvegarde automatiquement le **meilleur modèle** rencontré pendant l'entraînement (pas forcément le dernier epoch) | `monitor="val_accuracy"`, `save_best_only=True` |


In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,                 # tolère 5 epochs sans amélioration avant d'arrêter
        restore_best_weights=True,  # restaure les poids du meilleur epoch, pas du dernier
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,                 # divise le learning rate par 2 en cas de stagnation
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath="best_cnn_plant_disease.keras",
        monitor="val_accuracy",
        save_best_only=True,        # ne sauvegarde que si le modèle s'améliore
        verbose=1,
    ),
]

print("✅ Callbacks configurés : EarlyStopping, ReduceLROnPlateau, ModelCheckpoint")


## 🚀 PARTIE 4 — Entraînement

**Pourquoi `class_weight` ?**
L'étape 1 (EDA) a révélé un **déséquilibre entre classes**. Sans correction, le modèle aurait tendance à privilégier les classes majoritaires. On calcule donc un poids par classe (`compute_class_weight`, méthode `"balanced"`) — les classes rares reçoivent un poids plus élevé dans le calcul de la loss, ce qui force le modèle à leur accorder autant d'attention.

**Pourquoi `steps_per_epoch` et `validation_steps` ?**
Avec un générateur Keras qui boucle indéfiniment, on doit indiquer explicitement combien de batchs constituent un epoch complet : `nombre_images // batch_size`.

**Paramètres d'entraînement (cahier des charges) :** `epochs=20`, `batch_size=32` (déjà fixé dans les générateurs).


In [ ]:
# Calcul des poids de classes pour gérer le déséquilibre détecté à l'étape 1
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_generator.classes),
    y=train_generator.classes,
)
class_weight_dict = dict(enumerate(class_weights_array))

print("⚖️  Exemple de poids de classes calculés (5 premières) :")
for i in list(class_weight_dict.keys())[:5]:
    print(f"   Classe {i} ({class_names[i]:<30}) -> poids = {class_weight_dict[i]:.3f}")


In [ ]:
EPOCHS = 20

steps_per_epoch = train_generator.samples // BATCH_SIZE
validation_steps = validation_generator.samples // BATCH_SIZE

print(f"🚀 Démarrage de l'entraînement : {EPOCHS} epochs, batch_size={BATCH_SIZE}")
print(f"   Steps par epoch (train)      : {steps_per_epoch}")
print(f"   Steps de validation          : {validation_steps}")

start_time = time.time()

history = cnn_model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,   # gestion du déséquilibre de classes
    verbose=1,                         # affiche la progression complète epoch par epoch
)

training_time = time.time() - start_time
print(f"\n✅ Entraînement terminé en {training_time/60:.1f} minutes ({len(history.history['loss'])} epochs réellement exécutés).")


## 📊 PARTIE 5 — Évaluation

**Pourquoi évaluer sur le jeu de Test et pas seulement sur la Validation ?**
La Validation a été utilisée **pendant** l'entraînement (callbacks, choix d'hyperparamètres) — le modèle "l'a un peu vue indirectement". Le **Test** est resté totalement isolé : c'est la seule mesure fiable de la capacité de généralisation du modèle sur des données jamais vues.

**Métriques calculées :**
- **Accuracy** : proportion globale de prédictions correctes
- **Loss** : valeur de la fonction de perte sur le test
- **Precision** (macro) : capacité à éviter les faux positifs, moyenne sur les 38 classes
- **Recall** (macro) : capacité à détecter tous les cas positifs, moyenne sur les 38 classes
- **F1-Score** (macro) : moyenne harmonique Precision/Recall — métrique clé en cas de déséquilibre de classes


In [ ]:
# --- Évaluation globale (loss, accuracy) via model.evaluate ---
test_generator.reset()
test_loss, test_accuracy = cnn_model.evaluate(test_generator, verbose=1)

print(f"\n📊 Résultats sur le jeu de TEST :")
print(f"   Test Loss     : {test_loss:.4f}")
print(f"   Test Accuracy : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")


In [ ]:
# --- Prédictions détaillées pour Precision / Recall / F1 / Confusion Matrix ---
test_generator.reset()
y_pred_probs = cnn_model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)       # classe prédite = probabilité maximale
y_true = test_generator.classes                 # vraies classes (générateur non mélangé : shuffle=False)

precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("📊 Métriques détaillées sur le jeu de TEST :\n")
print(f"{'Métrique':<20}{'Macro avg':<15}{'Weighted avg':<15}")
print("-" * 50)
print(f"{'Precision':<20}{precision_macro:<15.4f}{precision_weighted:<15.4f}")
print(f"{'Recall':<20}{recall_macro:<15.4f}{recall_weighted:<15.4f}")
print(f"{'F1-Score':<20}{f1_macro:<15.4f}{f1_weighted:<15.4f}")


## 📈 PARTIE 6 — Visualisations

**Pourquoi ces 4 visualisations ?**
1. **Courbe Accuracy** (train vs val) → suit la progression de l'apprentissage
2. **Courbe Loss** (train vs val) → indicateur le plus fiable d'overfitting (si la loss val remonte alors que la loss train continue de baisser, c'est un signal d'overfitting)
3. **Matrice de confusion** → identifie précisément **quelles classes sont confondues entre elles** (ex : deux maladies aux symptômes visuellement proches)
4. **Classification Report** → détail Precision/Recall/F1 **par classe**, essentiel pour repérer les classes les plus difficiles à reconnaître


In [ ]:
# --- Courbes Accuracy et Loss ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(history.history["accuracy"], label="Train Accuracy", linewidth=2)
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy", linewidth=2)
axes[0].set_title("Évolution de l'Accuracy", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history["loss"], label="Train Loss", linewidth=2)
axes[1].plot(history.history["val_loss"], label="Validation Loss", linewidth=2)
axes[1].set_title("Évolution de la Loss", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("cnn_courbes_entrainement.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# --- Matrice de confusion (38x38, normalisée en %) ---
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype("float") / cm.sum(axis=1, keepdims=True) * 100

plt.figure(figsize=(20, 18))
sns.heatmap(
    cm_normalized,
    annot=False,             # 38x38 = trop dense pour annoter chaque case
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={"label": "% de prédictions"},
)
plt.title("Matrice de confusion (normalisée, en %) — CNN From Scratch", fontsize=15, fontweight="bold")
plt.xlabel("Classe prédite")
plt.ylabel("Classe réelle")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig("cnn_matrice_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

# Identification des paires de classes les plus confondues (hors diagonale)
cm_no_diag = cm_normalized.copy()
np.fill_diagonal(cm_no_diag, 0)
top_confusions_idx = np.dstack(np.unravel_index(np.argsort(-cm_no_diag.ravel()), cm_no_diag.shape))[0][:5]

print("🔍 Top 5 confusions entre classes :")
for true_idx, pred_idx in top_confusions_idx:
    if cm_no_diag[true_idx, pred_idx] > 0:
        print(f"   {class_names[true_idx]:<35} confondu avec {class_names[pred_idx]:<35} "
              f"({cm_no_diag[true_idx, pred_idx]:.1f}% des cas)")


In [ ]:
# --- Classification Report détaillé (par classe) ---
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()

print("📋 Classification Report (par classe) :\n")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# Sauvegarde du rapport pour le rapport PDF final
report_df.to_csv("cnn_classification_report.csv")
print("💾 Rapport sauvegardé : cnn_classification_report.csv")

# Visualisation des 10 classes avec le F1-score le plus FAIBLE (à surveiller)
worst_classes = report_df.iloc[:-3].sort_values("f1-score").head(10)   # on exclut les lignes "accuracy"/"macro avg"/"weighted avg"

plt.figure(figsize=(10, 6))
sns.barplot(x=worst_classes["f1-score"], y=worst_classes.index, hue=worst_classes.index, palette="Reds_r", legend=False)
plt.title("10 classes les plus difficiles à reconnaître (F1-score le plus faible)", fontsize=13, fontweight="bold")
plt.xlabel("F1-Score")
plt.tight_layout()
plt.savefig("cnn_classes_difficiles.png", dpi=150, bbox_inches="tight")
plt.show()


## 💾 PARTIE 7 — Sauvegarde du modèle

**Pourquoi sauvegarder dans 2 formats ?**
- **`.keras`** : format natif Keras 3 (recommandé), contient l'architecture + les poids + la configuration d'entraînement dans un seul fichier optimisé.
- **`.h5`** (HDF5) : format historique, encore largement utilisé pour la **compatibilité** avec d'anciens outils, scripts de déploiement, ou démonstrations (ex : Streamlit, Hugging Face Spaces).

On sauvegarde le modèle final (poids restaurés au meilleur epoch grâce à `restore_best_weights=True` d'EarlyStopping).


In [ ]:
# Sauvegarde au format natif Keras 3 (recommandé)
cnn_model.save("cnn_plant_disease.keras")
print("💾 Modèle sauvegardé : cnn_plant_disease.keras")

# Sauvegarde au format HDF5 (compatibilité avec d'anciens outils / déploiement)
cnn_model.save("cnn_plant_disease.h5")
print("💾 Modèle sauvegardé : cnn_plant_disease.h5")

# Sauvegarde de l'historique d'entraînement (utile pour comparer avec les futurs modèles - benchmarking étape 4)
with open("cnn_training_history.json", "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)
print("💾 Historique d'entraînement sauvegardé : cnn_training_history.json")

# Sauvegarde des métriques finales (pour le tableau de benchmarking de l'étape 4)
final_metrics = {
    "model_name": "CNN_From_Scratch",
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "precision_macro": float(precision_macro),
    "recall_macro": float(recall_macro),
    "f1_macro": float(f1_macro),
    "precision_weighted": float(precision_weighted),
    "recall_weighted": float(recall_weighted),
    "f1_weighted": float(f1_weighted),
    "total_params": int(total_params),
    "training_time_minutes": round(training_time / 60, 2),
    "epochs_executed": len(history.history["loss"]),
}
with open("cnn_final_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)
print("💾 Métriques finales sauvegardées : cnn_final_metrics.json (réutilisées pour le benchmarking de l'étape 4)")


## 🔎 PARTIE 8 — Analyse complète

**Pourquoi cette analyse ?**
Le cahier des charges (et un rapport OFPPT en général) exige une **lecture critique des résultats**, pas seulement des chiffres. La cellule de code ci-dessous **calcule automatiquement** des indicateurs objectifs (écart train/val, tendance de la loss...) afin de fonder l'analyse sur des données plutôt que des impressions.


In [ ]:
# --- Calcul automatique d'indicateurs d'overfitting ---
final_train_acc = history.history["accuracy"][-1]
final_val_acc = history.history["val_accuracy"][-1]
acc_gap = final_train_acc - final_val_acc

final_train_loss = history.history["loss"][-1]
final_val_loss = history.history["val_loss"][-1]

# Tendance de la val_loss sur les 5 derniers epochs (remontée = signal d'overfitting)
last_n = min(5, len(history.history["val_loss"]))
val_loss_trend = history.history["val_loss"][-last_n] - history.history["val_loss"][-1]

print("=" * 70)
print("DIAGNOSTIC AUTOMATIQUE")
print("=" * 70)
print(f"Accuracy finale Train       : {final_train_acc:.4f}")
print(f"Accuracy finale Validation  : {final_val_acc:.4f}")
print(f"Écart Train - Validation    : {acc_gap:.4f}")
print(f"Loss finale Train           : {final_train_loss:.4f}")
print(f"Loss finale Validation      : {final_val_loss:.4f}")

if acc_gap > 0.10:
    print("\n⚠️ DIAGNOSTIC : Écart important (>10%) entre Train et Validation -> OVERFITTING probable.")
elif acc_gap > 0.05:
    print("\n🟡 DIAGNOSTIC : Léger écart (5-10%) entre Train et Validation -> overfitting modéré, à surveiller.")
else:
    print("\n✅ DIAGNOSTIC : Écart faible (<5%) entre Train et Validation -> bonne généralisation.")


### 📝 Synthèse de l'analyse (à compléter avec les chiffres obtenus après exécution)

**✅ Points forts**
- Architecture progressive (32→64→128→256 filtres) cohérente avec la complexité croissante des motifs à apprendre.
- Utilisation de `BatchNormalization` à chaque bloc → entraînement stabilisé, convergence plus rapide.
- `Dropout(0.5)` + `class_weight` équilibré → deux mécanismes de régularisation actifs contre l'overfitting et le déséquilibre de classes.
- Callbacks complets (`EarlyStopping`, `ReduceLROnPlateau`, `ModelCheckpoint`) → entraînement robuste, pas de surentraînement inutile, meilleur modèle automatiquement conservé.
- Pipeline entièrement reproductible (générateurs et splits identiques à l'étape 2).

**⚠️ Points faibles attendus pour un CNN From Scratch (face à un modèle pré-entraîné)**
- Pas de transfert d'apprentissage : le modèle doit apprendre **toutes** les caractéristiques visuelles depuis zéro avec "seulement" ~54 000 images, contre des millions d'images vues par MobileNetV2/ResNet50 sur ImageNet.
- Risque de confusion plus élevé entre classes visuellement proches (cf. section "Top 5 confusions" ci-dessus) — un modèle pré-entraîné dispose généralement de représentations plus fines.
- Sensibilité accrue aux hyperparamètres (learning rate, architecture) en l'absence de poids déjà optimisés.

**🔍 Overfitting**
- Se référer au diagnostic automatique ci-dessus (écart Train/Validation). Si overfitting détecté : envisager d'augmenter le `Dropout`, réduire la complexité du modèle (moins de filtres), ou renforcer la Data Augmentation.

**💡 Recommandations pour la suite du projet**
1. Comparer ces résultats à ceux du Fine-Tuning MobileNetV2/ResNet50 (étape 4 — Benchmarking) : un modèle pré-entraîné devrait surpasser ce CNN From Scratch en accuracy ET en vitesse de convergence (moins d'epochs nécessaires).
2. Si l'écart Train/Validation est important, ajouter une régularisation L2 (`kernel_regularizer`) sur les couches denses.
3. Étudier en priorité les classes identifiées dans le graphique "classes les plus difficiles" — éventuellement collecter plus d'exemples ou affiner l'augmentation pour ces classes spécifiques.
4. Conserver ce CNN From Scratch comme **baseline de référence** dans le rapport final, pour démontrer objectivement l'apport du Transfer Learning.

---

### ➡️ Prochaine étape (Étape 4/9)
**Fine-Tuning de MobileNetV2**, en réutilisant les générateurs `mobilenet_train_gen` / `mobilenet_val_gen` / `mobilenet_test_gen` de l'étape 2, et en comparant les métriques sauvegardées dans `cnn_final_metrics.json` avec celles du nouveau modèle.

Dis-moi les résultats obtenus (accuracy, overfitting détecté ou non) et je préparerai le notebook de Fine-Tuning MobileNetV2.
